# Pipeline PCB — Extraction complète depuis un PDF

Pipeline complet : PDF → classification des pages → extraction des composants → affichage interactif.

**Scripts sources :**
- **Victoire** : `notebook/NotebookEP.ipynb` — liaison carte/schéma (code exact)
- **Rayane** : `parts/Lecture_des_composants/extract_easyocr.py` — extraction carte PCB (code exact)
- **Schéma** : `parts/Lecture_des_composants/extract_schema.py` — extraction schéma (code exact)
- **Ilyas** : `parts/Lecture_table/test1.py` — lecture BOM (code exact)
- **Maiwenn** : `parts/Extraction_schemas/prediction.py` — YOLO classification pages
- **Louis**: 'parts/extraction_noms_composants' - Yolo model

**Ordre d'exécution :** Cellule 1 → 2 → 3 → 4 → 5 → Pipeline


## Cellule 1 — Imports & configuration

In [12]:
%matplotlib widget
import os, sys, re, csv, io, json, time
from pathlib import Path
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from matplotlib.image import imread

# ── CHEMINS DES MODÈLES ──────────────────────────────────────────────────
# Modèle YOLO pour la CLASSIFICATION des pages (carte / schéma / table)
# Source : parts/Extraction_schemas/prediction.py
MODEL_CLASSIFICATION = Path('../../models/extraction/best.pt')

# Modèle YOLO pour la DÉTECTION des composants (carte PCB)
# Source : parts/Lecture_table/test1.py  (PATH_TO_YOLO_MODEL)
MODEL_COMPOSANTS = Path('models/component_names/nom_composants_v4/weights/best.pt')

# Répertoire de sortie
OUTPUT_DIR = Path('results/pipeline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✔  Imports OK')
print(f"   Modèle classification : {'✔ trouvé' if MODEL_CLASSIFICATION.exists() else '✘ MANQUANT — classification manuelle activée'}")
print(f"   Modèle composants     : {'✔ trouvé' if MODEL_COMPOSANTS.exists() else '✘ MANQUANT'}")


✔  Imports OK
   Modèle classification : ✘ MANQUANT — classification manuelle activée
   Modèle composants     : ✘ MANQUANT


## Cellule 2 — Partie Victoire (`notebook/NotebookEP.ipynb`, cellules 8/10/12/14/16 — code exact)

In [13]:
# ============================================================
# PARTIE VICTOIRE — code exact de notebook/NotebookEP.ipynb
# Cellules 8, 10, 12, 14, 16
# ============================================================

import csv
import re
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.image import imread
from matplotlib.animation import FuncAnimation

# SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP SET-UP
def transformer_csv(input_file: str, output_file: str) :
    # Opening the original file (output from step 2)
    with open(input_file, mode="r", newline="", encoding="utf-8") as infile:
        reader = csv.DictReader(infile)
        # Header of the output csv file
        fieldnames = ["identifiant", "x", "y"]
        # Opening the output file
        with open(output_file, mode="w", newline="", encoding="utf-8") as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            for row in reader:
                name = row["name"] # the identifiers' row in the input csv are labelled "name" in the header
                left = float(row["left"])
                top = float(row["top"])
                width = float(row["width"])
                height = float(row["height"])
                x = left + (width / 2)
                y = top + (height / 2)
                # writing in the output file
                writer.writerow({
                    "identifiant": name,
                    "x": x,
                    "y": y
                })


# STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1 STEP 3.1

def modif(filepathSource, filepathDest):
    output_rows = [] # dictionnary that contains the cleand rows

    # Pattern : expression that tolerates identifiers of a certains form (letters before numbers that may be separated by commas and/or blank spaces)
    pattern = r'([Cĉƈ<ç©¢ςĆČƇʗζㄈⓒⒸ©ćĈĊċč๔๕€ÇRŕ®ŗřʀŔŖƦЯГ尺ⓡⓇŔŕŖŗŘгѓQҩԳφ٩ҨⓠⓆъьвɓ฿βßƁ乃ⓑⒷţτťŧтŢŤŦ†ｲⓣⓉś§$ŝşѕŚŞŠらⓢⓈ§ŚśŜŝŞşѕĺľŀłℓŁĿĹŁしⓛⓁ£¦ĹĺĻļĽľĿŀŁđδðďÐĎƊのⓓⒹĎďĐđδ₫])\s*(\d+(?:\s*,\s*\d+)*)'

    # Opening the input CSV file
    with open(filepathSource, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')

        for row in reader: # we go through every row of the input csv file
            if len(row) < 3:
                continue  # Ignoring rows with missing values

            # suppressing blanck spaces from identifiers ; storing them into raw_text, x and y
            raw_text = row[0].strip()
            x = row[1].strip()
            y = row[2].strip()

            # Special case 1 : if the identifier begins with 5, we substituate it to a C
            if re.match(r'^5\s*\d+(?:\s*,\s*\d+)*$', raw_text):
                raw_text = re.sub(r'^5\s*', 'C', raw_text)

            # Special case 2 : if the identifier contains an i, we substituate it to a 1
            raw_text = re.sub(r'i', '1', raw_text, flags=re.IGNORECASE)

            # Searching for valid patterns into our blank free identifiers
            matches = re.findall(pattern, raw_text) # dictionnary
            # ex. matches = [('C', '1,2,3'), ('R', '5'), ('Q', '7')]

            for letter, number_seq in matches: #going through the (letter,number_seq) elements of the dictionnary

                # Interpretating special characters as the corresponding usual letter
                if re.match(r'[Cĉƈ<ç©¢ςĆČƇʗζㄈⓒⒸ©ćĈĊċč๔๕€Ç]', letter, flags=re.IGNORECASE):
                    letter = 'C' # redefine letter as its corrected version
                elif re.match(r'[Rŕ®ŗřʀŔŖƦЯГ尺ⓡⓇŔŕŖŗŘгѓ]', letter, flags=re.IGNORECASE):
                    letter = 'R'
                elif re.match(r'[ҩԳφ٩ҨⓠⓆ]', letter, flags=re.IGNORECASE):
                    letter = 'Q'
                elif re.match(r'[ъьвɓ฿βßƁ乃ⓑⒷ]', letter, flags=re.IGNORECASE):
                    letter = 'B'
                elif re.match(r'[ţτťŧтŢŤŦ†ｲⓣⓉ]', letter, flags=re.IGNORECASE):
                    letter = 'T'
                elif re.match(r'[ś§$ŝşѕŚŞŠらⓢⓈ§ŚśŜŝŞşѕ]', letter, flags=re.IGNORECASE):
                    letter = 'S'
                elif re.match(r'[ĺľŀłℓŁĿĹŁしⓛⓁ£¦ĹĺĻļĽľĿŀŁ]', letter, flags=re.IGNORECASE):
                    letter = 'L'
                elif re.match(r'[đδðďÐĎƊのⓓⒹĎďĐđδ₫]', letter, flags=re.IGNORECASE):
                    letter = 'D'
                else:
                    letter = letter.upper() #casting everything into uppercast

                # Cleaning numbers, including commas and blanck spaces
                numbers = [] # contains the numbers (generally 1) linked to the letter of the identifier
                for n in number_seq.split(','):
                    cleaned = n.strip()  # remove surrounding spaces
                    if cleaned.isdigit():  # keep only if it's a valid integer string
                        numbers.append(cleaned)

                # The identifier is build by concatenating the normalized letter (C, R, Q...) with the corresponding numbers
                # ex. letter "C" and num "2" -> label "C2"
                for num in numbers:
                    label = f"{letter}{num}"
                    output_rows.append([label, x, y]) # recreating the row by adding the memorized coordinates

    # Writing into the output file
    with open(filepathDest, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerow(['Identifiant', 'x', 'y'])
        writer.writerows(output_rows)


# STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2 STEP 3.2

def fusionner_csv(circuit_file, carte_file, output_file):
    circuit_data = { } #dictionnary : ex. {"C1": (150, 200), "R5": (300, 400)}
    carte_data = { } # idem

    # Reading the circuit file
    with open(circuit_file, newline='', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=',')
        for row in reader:
            if len(row) >= 3: # ignoring lines tht aren't conform
                key = row[0].strip() #identifier
                x = row[1].strip()
                y = row[2].strip()
                circuit_data[key] = (x, y)

    # Reading the board file
    with open(carte_file, newline='', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=',')
        for row in reader:
            if len(row) >= 3:
                key = row[0].strip()
                x = row[1].strip()
                y = row[2].strip()
                carte_data[key] = (x, y)

    # Fusionning data based on common keys (circuit / board)
    # Collect all keys from both dictionaries
    circuit_keys = set(circuit_data.keys())
    carte_keys = set(carte_data.keys())
    # Merge them
    combined_keys = circuit_keys.union(carte_keys)
    # Sort the result
    all_keys = sorted(combined_keys)

    merged_rows = [] # will contain the output rows
    for key in all_keys:
        x_circ, y_circ = circuit_data.get(key, ("", ""))
        # If the key exists -> we get the corresponding pair (ex. (150, 200)).
        # If the key doesn't exist -> we return the blank pair ("", "").
        x_cart, y_cart = carte_data.get(key, ("", ""))

        merged_rows.append([ #concatenating the data as a row that will be written in the output file
            key,
            x_circ, y_circ,
            x_cart, y_cart
        ])

    # Writing in the output file
    with open(output_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=',')
        writer.writerow([
            'Identifiant',
            'x_circ', 'y_circ',
            'x_cart', 'y_cart'
        ])
        writer.writerows(merged_rows)


# STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3 STEP 3.3

def afficher_carte_et_schema(carte_img_path, schema_img_path, csv_liaison_path):
    # Loading both images (board et scheme)
    carte = imread(carte_img_path)
    schema = imread(schema_img_path)

    # Lists storing the coordinates
    carte_pts = []
    schema_pts = []

    # Reading the linking .csv file
    with open(csv_liaison_path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=',')
        next(reader, None)  # Ignores the header
        for row in reader:
            if len(row) < 5:
                continue
            try:
                x_circ = float(row[1])
                y_circ = float(row[2])
                x_cart = float(row[3])
                y_cart = float(row[4])

                carte_pts.append([x_cart, y_cart])
                schema_pts.append([x_circ, y_circ])
            except ValueError:
                continue

    # Converts lists into NumPy tables
    carte_pts_np = np.array(carte_pts, dtype=float)
    schema_pts_np = np.array(schema_pts, dtype=float)

    if carte_pts_np.size == 0 or schema_pts_np.size == 0:
        raise ValueError("Aucun point valide trouvé dans le CSV.")

    def trouve_point_proche(x, y, points):
        d = np.sqrt((points[:, 0] - x)**2 + (points[:, 1] - y)**2)
        return np.argmin(d)

    # Prepares the displaying of both the board and the scheme
    fig, (ax_carte, ax_schema) = plt.subplots(1, 2, figsize=(12, 6))
    ax_carte.imshow(carte)
    ax_carte.axis("off")
    ax_schema.imshow(schema)
    ax_schema.axis("off")

    ani = None

    def ease_in_out(t):
        return 0.5 * (1 - np.cos(np.pi * t))

    def anime_zoom(xs, ys, zoom=100, frames=20, interval=30, pause_ms=1000):
        nonlocal ani

        xlim_init = ax_schema.get_xlim()
        ylim_init = ax_schema.get_ylim()

        xlim_target = (xs - zoom, xs + zoom)
        ylim_target = (ys + zoom, ys - zoom)

        pause_frames = int(pause_ms / interval)

        t = np.linspace(0, 1, frames)
        t_eased = ease_in_out(t)

        xlims = xlim_init[0] + (xlim_target[0] - xlim_init[0]) * t_eased
        xlims2 = xlim_init[1] + (xlim_target[1] - xlim_init[1]) * t_eased
        ylims = ylim_init[0] + (ylim_target[0] - ylim_init[0]) * t_eased
        ylims2 = ylim_init[1] + (ylim_target[1] - ylim_init[1]) * t_eased

        xlims = np.concatenate([xlims, np.full(pause_frames, xlim_target[0])])
        xlims2 = np.concatenate([xlims2, np.full(pause_frames, xlim_target[1])])
        ylims = np.concatenate([ylims, np.full(pause_frames, ylim_target[0])])
        ylims2 = np.concatenate([ylims2, np.full(pause_frames, ylim_target[1])])

        t_back = np.linspace(0, 1, frames)
        t_back_eased = ease_in_out(t_back)

        xlims = np.concatenate([xlims, xlim_target[0] + (xlim_init[0] - xlim_target[0]) * t_back_eased])
        xlims2 = np.concatenate([xlims2, xlim_target[1] + (xlim_init[1] - xlim_target[1]) * t_back_eased])
        ylims = np.concatenate([ylims, ylim_target[0] + (ylim_init[0] - ylim_target[0]) * t_back_eased])
        ylims2 = np.concatenate([ylims2, ylim_target[1] + (ylim_init[1] - ylim_target[1]) * t_back_eased])

        def update(i):
            ax_schema.set_xlim(xlims[i], xlims2[i])
            ax_schema.set_ylim(ylims[i], ylims2[i])
            return ax_schema,

        ani = FuncAnimation(fig, update, frames=len(xlims), interval=interval, blit=False, repeat=False)
        plt.draw()

    def on_click(event):
        if event.inaxes != ax_carte:
            return
        if event.xdata is None or event.ydata is None:
            return
        x, y = event.xdata, event.ydata
        idx = trouve_point_proche(x, y, carte_pts_np)
        xs, ys = schema_pts_np[idx]
        anime_zoom(xs, ys, zoom=400, frames=30, interval=30, pause_ms=1000)

    fig.canvas.mpl_connect('button_press_event', on_click)
    plt.show()

print("✔  Fonctions Victoire chargées (transformer_csv, modif, fusionner_csv, afficher_carte_et_schema)")


✔  Fonctions Victoire chargées (transformer_csv, modif, fusionner_csv, afficher_carte_et_schema)


## Cellule 3 — Extraction carte PCB (`extract_easyocr.py` — Rayane, code exact)

In [14]:
# ============================================================
# PARTIE RAYANE — code exact de parts/Lecture_des_composants/extract_easyocr.py
# ============================================================

import cv2
import numpy as np
import easyocr
import json as _json
import csv as _csv
import re
import sys
import time
from pathlib import Path
from ultralytics import YOLO

# ── Terminal helpers ──────────────────────────────────────────────────────────
class C:
    RESET='\033[0m'; BOLD='\033[1m'; BLUE='\033[34m'; CYAN='\033[36m'
    GREEN='\033[32m'; YELLOW='\033[33m'; RED='\033[31m'; GRAY='\033[90m'; WHITE='\033[97m'

def _c(text, *codes): return ''.join(codes) + str(text) + C.RESET
def step(icon, label, value=''):
    val = f'  {C.GRAY}{value}{C.RESET}' if value else ''
    print(f'  {icon}  {C.BOLD}{label}{C.RESET}{val}')
def ok(label): print(f'  {_c("✔", C.GREEN)}  {label}')
def progress_bar(current, total, width=30):
    pct=current/total if total else 0; filled=int(width*pct)
    bar=_c('█'*filled,C.CYAN)+_c('░'*(width-filled),C.GRAY)
    sys.stdout.write(f'\r  {_c("⟳",C.CYAN)}  OCR  [{bar}]  {_c(f"{current}/{total}",C.WHITE)}  {_c(f"{pct*100:.0f}%",C.YELLOW)}')
    sys.stdout.flush()
    if current==total: sys.stdout.write('\n')

# ── EasyOCR (chargé une seule fois) ──────────────────────────────────────────
print(f'\n  {_c("⟳", C.CYAN)}  Chargement EasyOCR...', end='', flush=True)
_t0 = time.time()
READER = easyocr.Reader(['en','fr','de','es','it','nl','pt'], gpu=True, verbose=False)
print(f'  {_c("✔", C.GREEN)}  EasyOCR pret  {_c(f"({time.time()-_t0:.1f}s)", C.GRAY)}')

ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-_'
OCR_CONF_MIN = 0.15
YOLO_IMGSZ = 1280

# ── Correcteur OCR ───────────────────────────────────────────────────────────
PREFIX_FIXES = {'0': 'O', '1': 'I', '8': 'B', '6': 'G', '5': 'S'}
DIGIT_FIXES  = {
    'I': '1', 'J': '1', 'L': '1', '|': '1',
    'O': '0', 'Q': '0', 'D': '0',
    'G': '6', 'Z': '2', 'S': '5', 'B': '8', 'T': '7',
}

def fix_ocr_confusion(text: str) -> str:
    if not text: return text
    text = text.strip().upper().replace(' ', '')
    i = 0
    while i < len(text) and not text[i].isdigit(): i += 1
    if i == len(text): return text
    prefix_raw = text[:i]
    rest = text[i:]
    VALID_2L = {'IC','TR','VR','BC','ZD','SW','TP','FR','ST','CF','FL','RL','DL','LED','SCR','FET','MOV'}
    if (len(prefix_raw) >= 2 and prefix_raw[-1] == 'I'
            and prefix_raw not in VALID_2L
            and prefix_raw[:-1] + prefix_raw[-1] not in VALID_2L):
        prefix_raw = prefix_raw[:-1]; rest = '1' + rest
    if len(rest) > 1 and rest[-1].isalpha() and rest[-1] not in DIGIT_FIXES:
        suffix, digits_raw = rest[-1], rest[:-1]
    else:
        suffix, digits_raw = '', rest
    fixed_prefix = ''.join(PREFIX_FIXES.get(ch,ch) if ch.isdigit() else ch for ch in prefix_raw)
    fixed_digits = ''.join(ch if ch.isdigit() else DIGIT_FIXES.get(ch,ch) for ch in digits_raw)
    return fixed_prefix + fixed_digits + suffix

POST_CORRECTIONS = {
    'UC402':'IC402','UC405':'IC405','RSI9':'R519','RL26':'R126','S501':'C501',
    'B178':'BC178','BCI78':'BC178','BCI7':'BC17','CI7A':'C17A','LC41':'C41',
    'ECN8':'C8','IB6':'B6','REO7':'R07','JEC740':'C740','STLO1':'ST01',
    'R62I':'R621','C41J':'C411','C50G':'C506','R5122':'R512','BC1788':'BC178',
    'BC178B':'BC178','T5502':'T502','R578':'R578',
}

def apply_post_corrections(text: str) -> str:
    key = text.strip().upper().replace(' ', '')
    return POST_CORRECTIONS.get(key, text)

def merge_nearby_boxes(boxes, gap_ratio=0.8, overlap_y_ratio=0.4):
    if not boxes: return boxes
    boxes = sorted(boxes, key=lambda b: b[0])
    merged = list(boxes); changed = True
    while changed:
        changed = False; new_merged = []; used = [False]*len(merged)
        for i in range(len(merged)):
            if used[i]: continue
            b1 = merged[i]; x1a,y1a,x2a,y2a,ca = b1
            best_j = -1; best_gap = float('inf')
            for j in range(i+1, len(merged)):
                if used[j]: continue
                b2=merged[j]; x1b,y1b,x2b,y2b,cb=b2
                if x1b < x2a: continue
                h_avg = ((y2a-y1a)+(y2b-y1b))/2; gap=x1b-x2a
                if gap > gap_ratio*h_avg: continue
                overlap_y=min(y2a,y2b)-max(y1a,y1b); min_h=min(y2a-y1a,y2b-y1b)
                if min_h==0 or overlap_y/min_h < overlap_y_ratio: continue
                if gap < best_gap: best_gap=gap; best_j=j
            if best_j >= 0:
                b2=merged[best_j]; x1b,y1b,x2b,y2b,cb=b2
                fused=(min(x1a,x1b),min(y1a,y1b),max(x2a,x2b),max(y2a,y2b),(ca+cb)/2)
                new_merged.append(fused); used[i]=used[best_j]=True; changed=True
            else:
                new_merged.append(b1); used[i]=True
        merged = new_merged
    return merged

def preprocess_crop(crop_bgr, scale=5):
    h,w=crop_bgr.shape[:2]; big=cv2.resize(crop_bgr,(w*scale,h*scale),interpolation=cv2.INTER_CUBIC)
    k=np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]); sharp=cv2.filter2D(big,-1,k)
    gray=cv2.cvtColor(sharp,cv2.COLOR_BGR2GRAY); clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    enhanced=clahe.apply(gray); bgr=cv2.cvtColor(enhanced,cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr,10,10,10,10,cv2.BORDER_CONSTANT,value=(255,255,255))

COMPONENT_PATTERN = re.compile(
    r'^([A-Z]{1,2}\s?\d{3,5}[A-Z]?|[A-Z]{1,3}\d{2,5}[A-Z]?|[A-Z]{2,4}\d{1,4}[A-Z]?|St\s?\d{3}|[A-Z]{1,3}\d+[-_]\d+)$',
    re.IGNORECASE)

def is_valid_ref(text: str) -> bool: return bool(COMPONENT_PATTERN.match(text.strip()))

def _run_easyocr(img_bgr):
    try:
        results = READER.readtext(img_bgr, detail=1, paragraph=False, allowlist=ALLOWLIST, width_ths=0.7)
    except Exception: return []
    out = []
    for (_,text,conf) in results:
        if conf < OCR_CONF_MIN: continue
        t=text.strip().upper(); t=re.sub(r'\s+','',t)
        if t: out.append((t,conf))
    return sorted(out, key=lambda x: x[1], reverse=True)

def preprocess_crop_otsu(crop_bgr, scale=5):
    h,w=crop_bgr.shape[:2]; big=cv2.resize(crop_bgr,(w*scale,h*scale),interpolation=cv2.INTER_CUBIC)
    gray=cv2.cvtColor(big,cv2.COLOR_BGR2GRAY); _,bw=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    bgr=cv2.cvtColor(bw,cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr,10,10,10,10,cv2.BORDER_CONSTANT,value=(255,255,255))

def run_ocr(crop_bgr, scale=5):
    def best_from(img):
        hits=_run_easyocr(img)
        if not hits: return "",False,0.0
        for (t,c) in hits:
            fixed=apply_post_corrections(fix_ocr_confusion(t))
            if is_valid_ref(fixed): return fixed,True,c
        fixed=apply_post_corrections(fix_ocr_confusion(hits[0][0]))
        return fixed,False,hits[0][1]
    pre_clahe=preprocess_crop(crop_bgr,scale=scale); h,w=pre_clahe.shape[:2]
    text,valid,conf=best_from(pre_clahe)
    if valid: return text,valid,conf
    if h>w*1.3 or not text:
        t2,v2,c2=best_from(cv2.rotate(pre_clahe,cv2.ROTATE_90_CLOCKWISE))
        if v2 or (t2 and c2>conf): text,valid,conf=t2,v2,c2
    if valid: return text,valid,conf
    pre_otsu=preprocess_crop_otsu(crop_bgr,scale=scale); t3,v3,c3=best_from(pre_otsu)
    if v3 or (t3 and c3>conf): return t3,v3,c3
    return text,valid,conf

def deduplicate_refs(detections):
    seen={}; result=[]
    for d in detections:
        if not d["is_valid_ref"]: result.append(d); continue
        key=d["text"].replace(' ','').upper(); score=d["yolo_conf"]*d["ocr_conf"]
        if key not in seen: seen[key]=len(result); result.append(d)
        else:
            existing=result[seen[key]]; existing_score=existing["yolo_conf"]*existing["ocr_conf"]
            if score>existing_score: result[seen[key]]=d
    return result

def upscale_image(img, factor: int):
    h,w=img.shape[:2]; big=cv2.resize(img,(w*factor,h*factor),interpolation=cv2.INTER_CUBIC)
    k=np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]); return cv2.filter2D(big,-1,k)

def auto_params(img):
    h,w=img.shape[:2]; gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); mean=float(gray.mean()); std=float(gray.std())
    dpi_est=int(w/(297/25.4))
    if dpi_est<80: pre_upscale=3
    elif dpi_est<150: pre_upscale=2
    else: pre_upscale=1
    eff_w,eff_h=w*pre_upscale,h*pre_upscale; eff_px=eff_w*eff_h
    if eff_px>3_000_000: imgsz,scale=1280,5
    elif eff_px>1_000_000: imgsz,scale=1024,6
    elif eff_px>300_000: imgsz,scale=640,8
    else: imgsz,scale=640,10
    if std<25: conf=0.25
    elif std<45: conf=0.30
    else: conf=0.40
    invert=mean<80
    params={"imgsz":imgsz,"scale":scale,"conf":conf,"invert":invert,"pre_upscale":pre_upscale}
    if dpi_est>=200: qual_tag=_c('● BONNE',C.GREEN)
    elif dpi_est>=120: qual_tag=_c('● MOYENNE',C.YELLOW)
    else: qual_tag=_c('● FAIBLE',C.RED)
    up_tag=(f'  {_c(f"→ pré-upscale ×{pre_upscale} ({eff_w}×{eff_h}px)",C.CYAN)}' if pre_upscale>1 else '')
    inv_tag=f'  {_c("⚠ fond sombre → inversion",C.YELLOW)}' if invert else ''
    print(f'  {_c("◈",C.CYAN)}  {_c("IMAGE",C.BOLD)}  {_c(f"{w}×{h} px",C.WHITE)}  {_c(f"~{dpi_est} DPI",C.CYAN)}  {qual_tag}{up_tag}{inv_tag}')
    print(f'  {_c("◈",C.CYAN)}  {_c("PARAMS",C.BOLD)}  imgsz={_c(imgsz,C.GREEN)}  scale={_c(scale,C.GREEN)}  conf={_c(conf,C.GREEN)}  contraste={_c(f"{std:.0f}",C.GRAY)}  lum={_c(f"{mean:.0f}",C.GRAY)}')
    return params

def extract_carte(img_path, model_path, conf_thresh=None):
    img=cv2.imread(str(img_path))
    if img is None: raise FileNotFoundError(f"Image introuvable : {img_path}")
    params=auto_params(img)
    if conf_thresh is not None: params["conf"]=conf_thresh
    if params["invert"]: img=cv2.bitwise_not(img)
    if params["pre_upscale"]>1:
        img=upscale_image(img,params["pre_upscale"]); ok(f'Pré-upscale ×{params["pre_upscale"]} → {img.shape[1]}×{img.shape[0]} px')
    t_yolo=time.time(); model=YOLO(str(model_path))
    yolo_result=model(img,conf=params["conf"],imgsz=params["imgsz"],verbose=False)[0]
    raw_boxes=yolo_result.boxes
    boxes_raw=[(int(b.xyxy[0][0]),int(b.xyxy[0][1]),int(b.xyxy[0][2]),int(b.xyxy[0][3]),float(b.conf[0])) for b in raw_boxes]
    boxes=merge_nearby_boxes(boxes_raw); fusions=len(boxes_raw)-len(boxes)
    step('◉','YOLO',f'{len(raw_boxes)} zones  →  {len(boxes)} apres merge ({_c(f"+{fusions} fusions",C.CYAN)})  {_c(f"{time.time()-t_yolo:.1f}s",C.GRAY)}')
    print()
    detections=[]; PAD=4
    for i,(x1,y1,x2,y2,conf_yolo) in enumerate(boxes):
        crop=img[max(0,y1-PAD):min(img.shape[0],y2+PAD),max(0,x1-PAD):min(img.shape[1],x2+PAD)]
        if crop.size==0: continue
        text,valid,ocr_conf=run_ocr(crop,scale=params["scale"])
        detections.append({"id":i+1,"text":text,"is_valid_ref":valid,"yolo_conf":round(conf_yolo,3),"ocr_conf":round(ocr_conf,3),"score":round(conf_yolo*ocr_conf,3),"bbox":{"x1":x1,"y1":y1,"x2":x2,"y2":y2},"center":{"x":(x1+x2)//2,"y":(y1+y2)//2}})
        progress_bar(i+1,len(boxes))
    before=sum(1 for d in detections if d["is_valid_ref"]); detections=deduplicate_refs(detections); after=sum(1 for d in detections if d["is_valid_ref"])
    if before!=after: ok(f'Deduplication : {before} → {_c(after,C.GREEN,C.BOLD)} refs uniques')
    return img, detections

print("✔  Fonctions Rayane chargées (extract_carte, fix_ocr_confusion, run_ocr, ...)")



  ⟳  Chargement EasyOCR...  ✔  EasyOCR pret  (2.4s)
✔  Fonctions Rayane chargées (extract_carte, fix_ocr_confusion, run_ocr, ...)


## Cellule 4 — Extraction schéma (`extract_schema.py`, code exact)

In [15]:
# ============================================================
# SCHÉMA — code exact de parts/Lecture_des_composants/extract_schema.py
# READER déjà chargé par la cellule 3 (Rayane)
# ============================================================

import pytesseract

SCHEMA_PATTERN = re.compile(r'^([A-Z]{1,3}\d{1,4}[A-Z]?)$', re.IGNORECASE)
STOP_WORDS = {
    'CHANNEL','RIGHT','LEFT','FILTER','INPUT','OUTPUT',
    'GND','VCC','VDD','VSS','PWR','OUT','IN','NC','COM',
    'AMP','OSC','REF','CLK','RST','INT','EXT',
    'REC','PLAY','STOP','AUX','TAPE','DISC','FLAY',
    'DIAGRAM','SCHEMATIC',
}

def fix_schema_ocr(text: str) -> str:
    t = text.strip().upper().replace(' ', '')
    if t and t[0] == 'O' and len(t) > 1 and t[1].isdigit():
        t = 'Q' + t[1:]
    i = 0
    while i < len(t) and not t[i].isdigit(): i += 1
    if i < len(t):
        prefix = t[:i]
        nums = ''.join('1' if c=='I' else '0' if c=='O' else c for c in t[i:])
        t = prefix + nums
    return t

def is_valid_schema(text: str) -> bool:
    t = text.strip().upper().replace(' ', '')
    if not t or len(t) < 2: return False
    if t in STOP_WORDS: return False
    if re.match(r'^\d', t): return False
    return bool(SCHEMA_PATTERN.match(t))

def remove_circuit_lines(img_bgr, scale: int):
    h,w = img_bgr.shape[:2]
    big = cv2.resize(img_bgr,(w*scale,h*scale),interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(big,cv2.COLOR_BGR2GRAY)
    _,bw = cv2.threshold(gray,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    line_len = max(40,scale*6)
    h_kern = cv2.getStructuringElement(cv2.MORPH_RECT,(line_len,1))
    v_kern = cv2.getStructuringElement(cv2.MORPH_RECT,(1,line_len))
    h_lines = cv2.morphologyEx(bw,cv2.MORPH_OPEN,h_kern,iterations=2)
    v_lines = cv2.morphologyEx(bw,cv2.MORPH_OPEN,v_kern,iterations=2)
    no_lines = cv2.subtract(bw,cv2.add(h_lines,v_lines))
    dilated = cv2.dilate(no_lines,np.ones((2,2)),iterations=1)
    return cv2.bitwise_not(dilated)

TESS_CONFIG = ('--psm 11 -c tessedit_char_whitelist='
               'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-')

def ocr_tesseract_schema(processed_gray, scale: int, conf_min: int = 10):
    data = pytesseract.image_to_data(processed_gray, config=TESS_CONFIG,
                                     output_type=pytesseract.Output.DICT)
    results = []
    for i, text in enumerate(data['text']):
        t = text.strip(); conf = int(data['conf'][i])
        if conf < conf_min or not t or len(t) < 2: continue
        cx = int((data['left'][i]+data['width'][i]/2)/scale)
        cy = int((data['top'][i]+data['height'][i]/2)/scale)
        x1 = int(data['left'][i]/scale); y1 = int(data['top'][i]/scale)
        x2 = int((data['left'][i]+data['width'][i])/scale)
        y2 = int((data['top'][i]+data['height'][i])/scale)
        results.append({'raw':t.upper(),'conf':conf/100.0,'cx':cx,'cy':cy,'x1':x1,'y1':y1,'x2':x2,'y2':y2})
    return results

def ocr_easyocr_schema(img_bgr, scale: int, conf_min: float = 0.20):
    h,w = img_bgr.shape[:2]
    big = cv2.resize(img_bgr,(w*scale,h*scale),interpolation=cv2.INTER_CUBIC)
    k = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]); sharp = cv2.filter2D(big,-1,k)
    try:
        results = READER.readtext(sharp,detail=1,paragraph=False,allowlist=ALLOWLIST,width_ths=0.9,min_size=5)
    except Exception: return []
    out = []
    for (bbox_pts,text,conf) in results:
        if conf < conf_min: continue
        t = text.strip().upper().replace(' ','')
        if not t: continue
        pts = np.array(bbox_pts)
        cx=int(pts[:,0].mean()/scale); cy=int(pts[:,1].mean()/scale)
        x1=int(pts[:,0].min()/scale); y1=int(pts[:,1].min()/scale)
        x2=int(pts[:,0].max()/scale); y2=int(pts[:,1].max()/scale)
        out.append({'raw':t,'conf':conf,'cx':cx,'cy':cy,'x1':x1,'y1':y1,'x2':x2,'y2':y2})
    return out

def deduplicate_schema(detections, pos_tol=25):
    kept = []
    for d in detections:
        dup = False
        for k in kept:
            if (abs(d['center']['x']-k['center']['x'])<pos_tol and
                    abs(d['center']['y']-k['center']['y'])<pos_tol and
                    d['text']==k['text']):
                dup=True; break
        if not dup: kept.append(d)
    seen={}; result_invalids=[]
    for d in kept:
        if not d['is_valid']: result_invalids.append(d); continue
        key=d['text'].upper().replace(' ','')
        if key not in seen or d['ocr_conf']>seen[key]['ocr_conf']: seen[key]=d
    return result_invalids + list(seen.values())

def extract_schema_img(img_path, scale=6, tess_conf_min=10):
    img = cv2.imread(str(img_path))
    if img is None: raise FileNotFoundError(f"Image introuvable : {img_path}")
    if img.shape[2] == 4: img = cv2.cvtColor(img,cv2.COLOR_BGRA2BGR)
    h,w = img.shape[:2]; gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); mean=float(gray.mean())
    dpi_est = int(w/(210/25.4))
    print(f'  Schema : {w}x{h} px  |  ~{dpi_est} DPI estimé')
    if mean < 80: img = cv2.bitwise_not(img)
    processed = remove_circuit_lines(img, scale)
    raw_tess = ocr_tesseract_schema(processed, scale, conf_min=tess_conf_min)
    raw_easy = []
    if len(raw_tess) < 20:
        raw_easy = ocr_easyocr_schema(img, scale=min(scale,5))
    all_raw = raw_tess + raw_easy
    detections = []
    for r in all_raw:
        fixed = fix_schema_ocr(r['raw']); valid = is_valid_schema(fixed)
        detections.append({'text':fixed,'raw':r['raw'],'is_valid':valid,'ocr_conf':round(r['conf'],3),
                           'source':'tess' if r in raw_tess else 'easy',
                           'bbox':{'x1':r['x1'],'y1':r['y1'],'x2':r['x2'],'y2':r['y2']},
                           'center':{'x':r['cx'],'y':r['cy']}})
    detections = deduplicate_schema(detections)
    valids = [d for d in detections if d['is_valid']]
    print(f'  Schema : {len(valids)} refs valides trouvées')
    return img, detections

print("✔  Fonctions schéma chargées (extract_schema_img, remove_circuit_lines, ...)")


✔  Fonctions schéma chargées (extract_schema_img, remove_circuit_lines, ...)


## Cellule 5 — Lecture BOM/Table (`parts/Lecture_table/test1.py`, code exact)

In [16]:
# ============================================================
# LECTURE TABLE — code exact de parts/Lecture_table/test1.py
# ============================================================

import pytesseract
from ultralytics import YOLO as _YOLO_TABLE

# ==========================================
# CONFIGURATION
# ==========================================
PATH_TO_YOLO_MODEL = str(MODEL_COMPOSANTS)  # chemin configuré cellule 1

try:
    yolo_model = _YOLO_TABLE(PATH_TO_YOLO_MODEL)
    print("✓ Modèle YOLO chargé.")
except Exception as e:
    print(f"Erreur chargement YOLO : {e}")
    yolo_model = None

# Liste des préfixes électroniques standards pour filtrer les erreurs OCR
PREFIXES_VALIDES = ['R','C','U','D','Q','L','J','SW','F','TP','RV','Y','DZ','IC']

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def clean_ocr_text(text):
    t = re.sub(r'[^A-Z0-9]', '', text.upper())
    found_prefix = None; remaining_part = ""
    for p in sorted(PREFIXES_VALIDES, key=len, reverse=True):
        match = re.search(r'('+p+r')([A-Z0-9]+)', t)
        if match:
            found_prefix=match.group(1); remaining_part=match.group(2); break
    if found_prefix:
        suffix=remaining_part.replace('S','5').replace('A','4').replace('O','0')
        suffix=suffix.replace('I','1').replace('L','1').replace('Z','2')
        suffix=suffix.replace('G','6').replace('B','8')
        num_match=re.search(r'\d+', suffix)
        if num_match:
            num_str=num_match.group().lstrip('0')
            if not num_str: num_str="0"
            return found_prefix+num_str
    return None

def preprocess_for_ocr(img, factor=3):
    if img.size==0: return img
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    gray=cv2.resize(gray,None,fx=factor,fy=factor,interpolation=cv2.INTER_CUBIC)
    return cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]

def detect_refs_tesseract(img):
    if img is None: return []
    h,w=img.shape[:2]; crop=img[:,:int(w*0.18)]; proc=preprocess_for_ocr(crop,factor=2)
    data=pytesseract.image_to_data(proc,config='--oem 3 --psm 11',output_type=pytesseract.Output.DICT)
    found_refs=[]; last_prefix="R"
    for i,txt in enumerate(data["text"]):
        clean=txt.strip().upper()
        if len(clean)<2: continue
        parts=re.split(r",",clean)
        for p in parts:
            m=re.match(r"([A-Z]+)(\d+)",p)
            if m:
                last_prefix=m.group(1)
                found_refs.append({"ref":last_prefix+m.group(2),"cx":data["left"][i]/2,"cy":data["top"][i]/2,"bbox":(data["left"][i]/2,data["top"][i]/2,(data["left"][i]+data["width"][i])/2,(data["top"][i]+data["height"][i])/2)})
            else:
                num=re.search(r"\d+",p)
                if num:
                    found_refs.append({"ref":last_prefix+num.group(),"cx":data["left"][i]/2,"cy":data["top"][i]/2,"bbox":(data["left"][i]/2,data["top"][i]/2,(data["left"][i]+data["width"][i])/2,(data["top"][i]+data["height"][i])/2)})
    return found_refs

def detect_refs_hybrid(img):
    if img is None or yolo_model is None: return []
    results=yolo_model.predict(source=img,conf=0.20,imgsz=1280,verbose=False)
    found_refs=[]
    for result in results:
        for box in result.boxes:
            b=box.xyxy[0].cpu().numpy().astype(int)
            roi=img[max(0,b[1]-5):b[3]+5,max(0,b[0]-5):b[2]+5]
            if roi.size>0:
                roi_proc=preprocess_for_ocr(roi,factor=4)
                config='--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                raw_text=pytesseract.image_to_string(roi_proc,config=config).strip()
                final_ref=clean_ocr_text(raw_text)
                if final_ref:
                    found_refs.append({"ref":final_ref,"cx":float((b[0]+b[2])/2),"cy":float((b[1]+b[3])/2),"bbox":(float(b[0]),float(b[1]),float(b[2]),float(b[3]))})
    return found_refs

print("✔  Fonctions table chargées (detect_refs_tesseract, detect_refs_hybrid, clean_ocr_text)")


Erreur chargement YOLO : [Errno 2] No such file or directory: 'models/component_names/nom_composants_v4/weights/best.pt'
✔  Fonctions table chargées (detect_refs_tesseract, detect_refs_hybrid, clean_ocr_text)


## Cellule 6 — Classification des pages PDF (`prediction.py`, code exact)

In [17]:
# ============================================================
# CLASSIFICATION DES PAGES — basé sur parts/Extraction_schemas/prediction.py
# ============================================================

from pdf2image import convert_from_path
import numpy as np

_classif_model = None
if MODEL_CLASSIFICATION.exists():
    try:
        from ultralytics import YOLO as _YOLO_CLASSIF
        _classif_model = _YOLO_CLASSIF(str(MODEL_CLASSIFICATION))
        print(f'✔  Modele classification charge : {MODEL_CLASSIFICATION}')
    except Exception as e:
        print(f'⚠  Impossible de charger le modele classification : {e}')
        print('   Mode classification manuelle active')
else:
    print('⚠  Modele classification non trouve (models/extraction/best.pt)')
    print('   Mode classification manuelle active')

def classer_page_yolo(page_pil):
    if _classif_model is None:
        return None
    result = _classif_model.predict(source=page_pil, conf=0.212, verbose=False)
    classes_trouvees = []
    for box in result[0].boxes:
        cls = result[0].names[int(box.cls[0])]
        classes_trouvees.append(cls)
    for label in ['carte', 'schema', 'table']:
        if label in classes_trouvees:
            return label
    return classes_trouvees[0] if classes_trouvees else 'ignorer'

def charger_pdf(pdf_path, dpi=300):
    print(f'  Conversion PDF en images ({dpi} DPI)...')
    pages = convert_from_path(pdf_path, dpi=dpi)
    print(f'  {len(pages)} page(s) extraites')
    return pages

print('✔  Fonctions classification chargees (charger_pdf, classer_page_yolo)')


⚠  Modele classification non trouve (models/extraction/best.pt)
   Mode classification manuelle active
✔  Fonctions classification chargees (charger_pdf, classer_page_yolo)


## Pipeline — Charger un PDF, classifier, extraire et afficher

**Executer cette cellule pour lancer le pipeline complet.**

In [18]:
# ============================================================
# PIPELINE PRINCIPAL
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

_state = {
    'pages': [], 'pdf_path': None, 'classifs': {},
    'cartes_imgs': [], 'schemas_imgs': [], 'tables_imgs': [],
    'refs_carte': [], 'refs_schema': [], 'refs_table': [],
}

# ── Etape 1 : Charger un PDF ──────────────────────────────────────────────
out_upload = widgets.Output()
upload_btn = widgets.FileUpload(accept='.pdf', multiple=False, description='Choisir un PDF')
lbl_upload = widgets.Label('Aucun PDF charge')

def on_upload(change):
    with out_upload:
        clear_output()
        if not upload_btn.value:
            return
        fname = list(upload_btn.value.keys())[0]
        pdf_bytes = upload_btn.value[fname]['content']
        pdf_path = OUTPUT_DIR / fname
        pdf_path.write_bytes(pdf_bytes)
        lbl_upload.value = f'Charge : {fname}'
        _state['pdf_path'] = str(pdf_path)
        pages = charger_pdf(str(pdf_path), dpi=300)
        _state['pages'] = pages
        print(f'{len(pages)} page(s) pretes')
        fig, axes = plt.subplots(1, min(len(pages), 6), figsize=(min(len(pages)*3, 18), 4))
        if len(pages) == 1: axes = [axes]
        for ax, (i, pg) in zip(axes, enumerate(pages[:6])):
            ax.imshow(pg); ax.set_title(f'Page {i+1}'); ax.axis('off')
        plt.tight_layout(); plt.show()

upload_btn.observe(on_upload, names='value')
display(widgets.VBox([
    widgets.HTML('<b>Etape 1 — Charger un PDF</b>'),
    upload_btn, lbl_upload, out_upload
]))

# ── Etape 2 : Classifier les pages ───────────────────────────────────────
out_classif = widgets.Output()
btn_classif = widgets.Button(description='Classifier les pages', button_style='info')

def on_classif(b):
    with out_classif:
        clear_output()
        pages = _state['pages']
        if not pages:
            print('Chargez d abord un PDF (Etape 1)')
            return
        radios = []
        for i, pg in enumerate(pages):
            cls_auto = classer_page_yolo(pg)
            label_auto = cls_auto if cls_auto else 'ignorer'
            r = widgets.RadioButtons(
                options=['carte', 'schema', 'table', 'ignorer'],
                value=label_auto,
                description=f'Page {i+1}:',
                style={'description_width': '80px'}
            )
            radios.append(r)
            src = 'YOLO' if cls_auto else 'manuel'
            print(f'  Page {i+1} -> {label_auto} ({src})')
        btn_val = widgets.Button(description='Valider', button_style='success')
        display(widgets.VBox(radios + [btn_val]))
        def on_valider(b2):
            with out_classif:
                for i, r in enumerate(radios):
                    _state['classifs'][i] = r.value
                n = {v: sum(1 for c in _state['classifs'].values() if c==v) for v in ['carte','schema','table','ignorer']}
                print(f'Classifications validees : {n}')
        btn_val.on_click(on_valider)

btn_classif.on_click(on_classif)
display(widgets.VBox([
    widgets.HTML('<b>Etape 2 — Classifier les pages</b>'),
    btn_classif, out_classif
]))

# ── Etape 3 : Extraction ─────────────────────────────────────────────────
out_extract = widgets.Output()
btn_extract = widgets.Button(description='Lancer l extraction', button_style='warning')

def sauver_page_png(page_pil, nom):
    path = OUTPUT_DIR / nom
    page_pil.save(str(path))
    return str(path)

def detections_to_raw_csv(detections_carte, path):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['name','left','top','width','height'])
        w.writeheader()
        for d in detections_carte:
            if not d.get('is_valid_ref', False): continue
            b = d['bbox']
            w.writerow({'name': d['text'], 'left': b['x1'], 'top': b['y1'],
                        'width': b['x2']-b['x1'], 'height': b['y2']-b['y1']})

def detections_schema_to_csv(detections_schema, path):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['Identifiant','x','y'])
        w.writeheader()
        for d in detections_schema:
            if not d.get('is_valid', False): continue
            w.writerow({'Identifiant': d['text'],
                        'x': d['center']['x'], 'y': d['center']['y']})

def on_extract(b):
    with out_extract:
        clear_output()
        if not _state['classifs']:
            print('Classifiez d abord les pages (Etape 2)')
            return
        pages = _state['pages']
        _state.update({'cartes_imgs':[],'schemas_imgs':[],'tables_imgs':[],
                       'refs_carte':[],'refs_schema':[],'refs_table':[]})
        for i, cls in _state['classifs'].items():
            if cls == 'ignorer': continue
            pg = pages[i]
            png_path = sauver_page_png(pg, f'page_{i+1}_{cls}.png')
            if cls == 'carte':
                _state['cartes_imgs'].append(png_path)
                print(f'[Carte] Page {i+1} -> extraction YOLO+EasyOCR...')
                _, dets = extract_carte(png_path, MODEL_COMPOSANTS)
                _state['refs_carte'].extend(dets)
                print(f'  {sum(1 for d in dets if d["is_valid_ref"])} refs valides')
            elif cls == 'schema':
                _state['schemas_imgs'].append(png_path)
                print(f'[Schema] Page {i+1} -> extraction Tesseract+EasyOCR...')
                _, dets = extract_schema_img(png_path)
                _state['refs_schema'].extend(dets)
                print(f'  {sum(1 for d in dets if d["is_valid"])} refs valides')
            elif cls == 'table':
                _state['tables_imgs'].append(png_path)
                print(f'[Table] Page {i+1} -> lecture BOM...')
                img_t = cv2.imread(png_path)
                refs = detect_refs_tesseract(img_t)
                _state['refs_table'].extend(refs)
                print(f'  {len(refs)} refs trouvees')
        print(f'Extraction terminee — Carte:{len(_state["refs_carte"])} Schema:{len(_state["refs_schema"])} Table:{len(_state["refs_table"])}')

btn_extract.on_click(on_extract)
display(widgets.VBox([
    widgets.HTML('<b>Etape 3 — Extraction des composants</b>'),
    btn_extract, out_extract
]))

# ── Etape 4 : Affichage interactif (Victoire) ────────────────────────────
out_display = widgets.Output()
btn_display = widgets.Button(description='Afficher carte + schema', button_style='primary')

def on_display(b):
    with out_display:
        clear_output()
        if not _state['cartes_imgs'] or not _state['schemas_imgs']:
            print('Il faut au moins une carte ET un schema.')
            return
        carte_img  = _state['cartes_imgs'][0]
        schema_img = _state['schemas_imgs'][0]
        raw_csv   = str(OUTPUT_DIR / 'raw_carte.csv')
        coord_csv = str(OUTPUT_DIR / 'coord_carte.csv')
        clean_csv = str(OUTPUT_DIR / 'clean_carte.csv')
        schema_csv = str(OUTPUT_DIR / 'coord_schema.csv')
        liaison_csv = str(OUTPUT_DIR / 'liaison.csv')
        detections_to_raw_csv(_state['refs_carte'], raw_csv)
        transformer_csv(raw_csv, coord_csv)
        modif(coord_csv, clean_csv)
        detections_schema_to_csv(_state['refs_schema'], schema_csv)
        fusionner_csv(schema_csv, clean_csv, liaison_csv)
        print(f'CSVs generes : {liaison_csv}')
        afficher_carte_et_schema(carte_img, schema_img, liaison_csv)

btn_display.on_click(on_display)
display(widgets.VBox([
    widgets.HTML('<b>Etape 4 — Affichage interactif (cliquez la carte pour zoomer sur le schema)</b>'),
    btn_display, out_display
]))

# ── Etape 5 : Corrections proposees ──────────────────────────────────────
out_corr = widgets.Output()
btn_corr = widgets.Button(description='Voir les corrections proposees')

def on_corr(b):
    with out_corr:
        clear_output()
        refs_table  = {r['ref'] for r in _state['refs_table']}
        refs_carte  = {d['text'] for d in _state['refs_carte']  if d.get('is_valid_ref')}
        refs_schema = {d['text'] for d in _state['refs_schema'] if d.get('is_valid')}
        refs_images = refs_carte | refs_schema
        absents_images = refs_table - refs_images
        absents_table  = refs_images - refs_table
        commun = refs_table & refs_images
        print(f'Composants en commun (table + images) : {len(commun)}')
        if absents_images:
            print(f'Dans la table mais absents des images ({len(absents_images)}) :')
            print('  ' + ', '.join(sorted(absents_images)))
        if absents_table:
            print(f'Detectes sur images mais absents de la table ({len(absents_table)}) :')
            print('  ' + ', '.join(sorted(absents_table)))
        if not absents_images and not absents_table:
            print('Aucune divergence detectee.')

btn_corr.on_click(on_corr)
display(widgets.VBox([
    widgets.HTML('<b>Etape 5 — Corrections proposees</b>'),
    btn_corr, out_corr
]))


## Mode direct — Utiliser des images existantes (sans PDF)

Modifiez les chemins et executez cette cellule directement.

In [22]:
# ── Mode direct : modifiez les chemins selon vos fichiers ─────────────────
CARTE_PATH  = 'Lecture_des_composants/carte.png'
SCHEMA_PATH = 'Lecture_des_composants/schema_test.png'
TABLE_PATH  = 'Lecture_des_composants/table_test.png'

import os
for p in [CARTE_PATH, SCHEMA_PATH, TABLE_PATH]:
    print(f'  {"OK" if os.path.exists(p) else "MANQUANT"} : {p}')

raw_csv    = str(OUTPUT_DIR / 'raw_direct.csv')
coord_csv  = str(OUTPUT_DIR / 'coord_direct.csv')
clean_csv  = str(OUTPUT_DIR / 'clean_direct.csv')
schema_csv = str(OUTPUT_DIR / 'schema_direct.csv')
liais_csv  = str(OUTPUT_DIR / 'liaison_direct.csv')

# ── Extraction carte ──────────────────────────────────────────────────────
print('\n=== Extraction carte ===')
_, dets_carte = extract_carte(CARTE_PATH, 'Lecture_des_composants/best.pt')
valid_carte = [d for d in dets_carte if d['is_valid_ref']]
print(f'  {len(valid_carte)} refs valides : {[d["text"] for d in valid_carte[:10]]}')

# ── Extraction schema ─────────────────────────────────────────────────────
print('\n=== Extraction schema ===')
_, dets_schema = extract_schema_img(SCHEMA_PATH)
valid_schema = [d for d in dets_schema if d['is_valid']]
print(f'  {len(valid_schema)} refs valides : {[d["text"] for d in valid_schema[:10]]}')

# ── Lecture table ─────────────────────────────────────────────────────────
print('\n=== Lecture table ===')
img_table = cv2.imread(TABLE_PATH)
if img_table is None:
    print('  (pas de table)')
    refs_table = []
else:
    refs_table = detect_refs_tesseract(img_table)
    print(f'  {len(refs_table)} refs : {[r["ref"] for r in refs_table[:10]]}')

# ── Pipeline Victoire : genere les CSVs de liaison ────────────────────────
def _raw(dets, path):
    with open(path,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=['name','left','top','width','height'])
        w.writeheader()
        for d in dets:
            if not d.get('is_valid_ref',False): continue
            b=d['bbox']
            w.writerow({'name':d['text'],'left':b['x1'],'top':b['y1'],
                        'width':b['x2']-b['x1'],'height':b['y2']-b['y1']})

def _sch(dets, path):
    with open(path,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=['Identifiant','x','y'])
        w.writeheader()
        for d in dets:
            if not d.get('is_valid',False): continue
            w.writerow({'Identifiant':d['text'],'x':d['center']['x'],'y':d['center']['y']})

_raw(dets_carte, raw_csv)
transformer_csv(raw_csv, coord_csv)
modif(coord_csv, clean_csv)
_sch(dets_schema, schema_csv)
fusionner_csv(schema_csv, clean_csv, liais_csv)

# ── Corrections proposees ─────────────────────────────────────────────────
refs_t  = {r['ref'] for r in refs_table}
refs_c  = {d['text'] for d in dets_carte  if d.get('is_valid_ref')}
refs_s  = {d['text'] for d in dets_schema if d.get('is_valid')}
refs_img = refs_c | refs_s
print(f'\n=== Corrections proposees ===')
print(f'  Commun table+images : {len(refs_t & refs_img)}')
abs_img = refs_t - refs_img
abs_tab = refs_img - refs_t
if abs_img: print(f'  Dans table, absents images : {sorted(abs_img)}')
if abs_tab: print(f'  Detectes images, absents table : {sorted(abs_tab)}')

# ── Affichage interactif (Victoire) ───────────────────────────────────────
print(f'\n=== Affichage interactif ===')
try:
    afficher_carte_et_schema(CARTE_PATH, SCHEMA_PATH, liais_csv)
except ValueError as e:
    print(f'  Affichage impossible : {e}')
    print('  (le CSV de liaison est vide — verifiez que carte ET schema ont des refs valides)')


  OK : Lecture_des_composants/carte.png
  OK : Lecture_des_composants/schema_test.png
  OK : Lecture_des_composants/table_test.png

=== Extraction carte ===
  ◈  IMAGE  2480×1200 px  ~212 DPI  ● BONNE
  ◈  PARAMS  imgsz=1024  scale=6  conf=0.4  contraste=48  lum=225
  ◉  YOLO  185 zones  →  155 apres merge (+30 fusions)  0.6s

  ⟳  OCR  [██████████████████████████████]  155/155  100%
  ✔  Deduplication : 69 → 62 refs uniques
  62 refs valides : ['L15', 'R417', 'C407', 'SA401', 'C5076', 'C414', 'C507', 'BC170', 'R825', 'R517']

=== Extraction schema ===
  Schema : 693x992 px  |  ~83 DPI estimé
  Schema : 14 refs valides trouvées
  14 refs valides : ['XZ122', 'A4', 'C6', 'Q1', 'A5', 'V7', 'S35', 'Q2', 'D4', 'S9']

=== Lecture table ===
  2 refs : ['R30', 'R48']

=== Corrections proposees ===
  Commun table+images : 0
  Dans table, absents images : ['R30', 'R48']
  Detectes images, absents table : ['A4', 'A5', 'BC170', 'BC176', 'BC178', 'C13', 'C178', 'C34', 'C401', 'C405', 'C407', 'C410'